# Solutions — Null Handling

**Dataset:** `samples.bakehouse.sales_customers`, `samples.bakehouse.sales_franchises`, `samples.tpch.customer`

**Topics:** isNull, isNotNull, fillna, dropna, coalesce, when/otherwise

> Reference solutions for practice notebooks.
> **Try the problem yourself first** — only open this when you've made a genuine attempt.
>
> Each solution includes a `# Why:` comment explaining the approach chosen.

In [ ]:
from pyspark.sql import functions as F, types as T
customers  = spark.table("samples.bakehouse.sales_customers")
franchises = spark.table("samples.bakehouse.sales_franchises")
tpch_cust  = spark.table("samples.tpch.customer")

---
## Problem 1 — Count Nulls Per Column

In [ ]:
# Why: build a list comprehension to count nulls in every column, then assemble with createDataFrame.
total_rows = customers.count()
null_data = []
for col_name in customers.columns:
    null_count = customers.filter(F.col(col_name).isNull()).count()
    null_rate  = round(null_count / total_rows * 100, 2)
    null_data.append((col_name, null_count, null_rate))

result_1 = spark.createDataFrame(null_data, ["column_name", "null_count", "null_rate_pct"])

---
## Problem 2 — Filter Nulls

In [ ]:
# Why: summarise both filter results in one DataFrame so the shape matches the test expectation.
null_count     = customers.filter(F.col("state").isNull()).count()
not_null_count = customers.filter(F.col("state").isNotNull()).count()

result_2 = spark.createDataFrame(
    [("state is null", null_count), ("state is not null", not_null_count)],
    ["filter_type", "customer_count"]
)

---
## Problem 3 — Fill Nulls

In [ ]:
# Why: fillna with a dict replaces nulls in multiple columns in one call.
result_3 = customers.fillna({"state": "Unknown", "gender": "Not Specified"})

---
## Problem 4 — Coalesce for Null Fallback

In [ ]:
# Why: coalesce returns the first non-null from the list — cleaner than nested when/otherwise.
result_4 = (
    franchises.withColumn(
        "location_label",
        F.coalesce(F.col("district"), F.col("city"), F.lit("Unknown"))
    ).select("franchiseID", "name", "district", "city", "location_label")
)

---
## Problem 5 — Drop Rows with Nulls

In [ ]:
# Why: compute counts before and after, then assemble into a summary DataFrame.
before_count = tpch_cust.count()
after_count  = tpch_cust.dropna(subset=["c_name", "c_mktsegment"]).count()

result_5 = spark.createDataFrame(
    [("before dropna", before_count), ("after dropna", after_count)],
    ["step", "row_count"]
)